In [1]:
import sys
sys.modules['__main__'].__file__ = 'ipython'

import multiprocess as mp
from multiprocess import Pool, cpu_count
# mp.set_start_method('fork')

from pdb import set_trace as st
from pprint import pprint
from collections import defaultdict

import msgspec
from tqdm.auto import tqdm
import pandas as pd
import polars as pl
from pathlib import Path
import re 
import numpy as np
from json_repair import repair_json

from sutime import SUTime
sutime = SUTime(mark_time_ranges=True, include_range=True)

# from python_heideltime import Heideltime
# heideltime_parser = Heideltime()
# heideltime_parser.set_document_type("NEWS")

# del heideltime_parser

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

In [2]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()
    
    output = []

    print("Reading the json file...")
    with open(file_path, "rb") as file:
        data = file.read()

        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


def read_txt(file_path, lines=False):
    with open(file_path, "r") as f:
        if lines:
            return f.readlines()
        return f.read()

# Temporal Nobel Prize

## Read the corpus data

In [ ]:
DATA_PATH = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl")
train_temporal_jsonl = read_json(DATA_PATH, jsonl=True)

In [ ]:
new_train_jsonl = []

for ix, line in tqdm(enumerate(train_temporal_jsonl), total=len(train_temporal_jsonl)):
    
    positive_passages, negative_passages = line["positive_passages"], line["negative_passages"]
    
    temp = {}
    temp["query_id"] = line["query_id"]
    temp["query"] = line["query"]
    temp["positive_passages"] = []
    temp["negative_passages"] = []

    for item in positive_passages:
        pos_docid = item["docid"]
        pos_text = item["text"]
        temp["positive_passages"].append(
            {"docid": pos_docid, "text": pos_text}
        )

    for item in negative_passages:
        neg_docid = item["docid"]
        neg_text = item["text"]
        temp["negative_passages"].append(
            {"docid": neg_docid, "text": neg_text}
        )
        
    new_train_jsonl.append(temp)

In [ ]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]

In [ ]:
print(TemporalAnnotation.model_json_schema())

## VLLM generation

In [ ]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]


In [ ]:
print(TemporalAnnotation.model_json_schema())

In [ ]:
import os

# export VLLM_USE_V1=1
# export TOKENIZERS_PARALLELISM=0
os.environ["VLLM_USE_V1"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "0"


from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from vllm.distributed import cleanup_dist_env_and_memory

guided_decoding_params = GuidedDecodingParams(
    json=TemporalAnnotation.model_json_schema(),
)

sampling_params = SamplingParams(
    max_tokens=32768, 
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0,
    guided_decoding=guided_decoding_params
)

llm = LLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
    generation_config="auto",
    max_model_len=32768,  # Limit context window
    max_num_seqs=4,  # Limit batch size
    gpu_memory_utilization=0.95,
    disable_cascade_attn=True,  # Avoid gibberish output due to batch inference
    seed=42,
)

In [ ]:
prompt_template = """You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME's output for your reference. We also provide you with previously annotated positive and negative samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.
Your final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.
Your temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. 
You must follow the definitions and instructions below.

* Allen relations and descriptions:
- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?
- After: Did ‘Event A’ occur after ‘Event B’ without any overlap between the two events?
- Meets: Did ‘Event A’ end in the same time as ‘Event B’ began? Answer True or False.
- MetBy: Did ‘Event B’ end in the same time as ‘Event A’ began? Answer True or False.
- Overlaps: Did ‘Event A’ begin before ‘Event B’ and end before ‘Event B’ ended, with some overlap between the two events?
- OverlappedBy: Did ‘Event B’ begin before ‘Event A’ and end before ‘Event A’ ended, with some overlap between the two events?
- Starts: ‘Event A’ begin in the same time as ‘Event B’, but end before ‘Event B’ ended?
- StartedBy: Did ‘Event B’ begin in the same time as ‘Event A’, but end before ‘Event A’ ended?
- During: Did ‘Event A’ begin after ‘Event B’ began and end before ‘Event B’ ended, being entirely contained within ‘Event B’?
- Contains: Did ‘Event A’ begin before ‘Event B’ began and end after ‘Event B’ ended, entirely containing ‘Event B’?
- Finishes: Did ‘Event A’ begin after ‘Event B’ began and end in the same time as ‘Event B’?
- FinishedBy: Did ‘Event B’ begin after ‘Event A’ began and end in the same time as ‘Event A’?
- Equals: Did ‘Event A’ begin in the same time as ‘Event B’ and end in the same time as ‘Event B’?
- Empty: a special case for TemporalAnswer where there is no temporal expression.

* Temporal signals (examples of what can appear in queries or passages): before, prior to, until, after, following, since, in, on, as of, for duration, during, while, when, from...to..., between, by, up to, first, last, around, as soon as, as long as, for, over, all through, throughout, etc.

* Taxonomy of "positive_passages" and "negative_passages" with examples that you should generate:
- Explicit temporal constraints (i.e., always have clear temporal expressions that can be anchored to a specific datetime): "who won the state of Texas in 2008?"; "what kind of government does Iran have after 1979?".
- Implicit temporal constraints (i.e., events that cannot be anchored to specific datetime. The rule of thumb is: if there is date or month or year in the question, like "before the **2004** general election", it is not implicit temporal): "who was the president after JFK died?"; "what team did Michael Jordan play for after the Bulls?".
- TemporalAnswer (i.e., Questions that inquire the datetime of an event instead of a general question with temporal constraints, usually starts with "when" or "what/which + day/date/month/year". There is no temporal expression in it.): "what year did the Knicks win the championship?"; "when was the United Nations founded?".

* Instructions:
1. You must output one or multiple valid JSONs, delimited by a newline, strictly following this pydantic schema:
{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'positive_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Positive Passages', 'type': 'array'}, 'negative_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Negative Passages', 'type': 'array'}}, 'required': ['query_id', 'query', 'temporal', 'positive_passages', 'negative_passages'], 'title': 'TemporalAnnotation', 'type': 'object'}.

2. "query_id": must be assigned with the provided "query_id".

3. “query":
- A query is a provided sentence that may contain one or many temporal expressions.
- Please note that SUTIME only provides explicit temporal expressions for the query and they are not perfect; therefore, you must double-check them and further detect additional explicit and implicit temporal expressions such as events.
- You must extract all events that are anchored to specific datetime (e.g., “2000 FA Cup Final”, “the 2007 election”, etc.).
- The "temporal" field should be extracted as written from the query's text and they must be concise, such as: "in April, 1906", "2 July 2010", "2004 general election", etc.

4. "positive_passages":
- Natural, QA-style questions with diverse phrasing that seek information from the query. Each question must contain exactly one extractable temporal expression, logically align with one of the query's temporal expressions, yet be diverse in Allen relations.
- Based on the query's temporal expressions, you must generate "positive_passages" that cover all "TemporalQueryType", including 'Explicit', 'Implicit', and 'TemporalAnswer', when possible:
    - You MUST prioritise generating questions with explicit temporal constraints, like "in 2010", "from 2010 to 2015", etc. They must logically align with the query’s temporal expression(s), such as being equals, overlapping with, or being contained within the query’s temporal expression(s). If the query has multiple temporal expressions, the generated questions must logically align with at least one of them.
    - You should also prioritise generating questions with implicit temporal constraints, such as "after EVENT", "before EVENT". These must logically align with the query’s temporal expression(s).
    - For both explicit and implicit "TemporalQueryType" questions, you must not use phrases like what/which date/day/month/year/time or when etc., that inquire about time. You must not confuse this with TemporalAnswer questions.
    - You can also generate "TemporalAnswer" (asking for datetime/duration/time-range of an EVENT, e.g., "When did EVENT happen?", "What time did he arrive?"). For this type of question, you must not add any temporal expression and you must set: "TemporalQueryType": "TemporalAnswer", "allen_relation": "Empty", and "temporal": []. Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- "temporal" field: must extract all exact text spans of the temporal expressions that exist in the generated question (not normalized or paraphrased). Regarding explicit and implicit "TemporalQueryType", they must be concise temporal expressions as written, such as "after July 2010", "from 2012 to 2014", that are not just normalized dates. For instance, instead of "In 1906 he moved with his family to a farm", prefer the concise "In 1906" and keep prepositions or context words that anchor the time, e.g., 'from', 'in', 'after', etc. Regarding "TemporalAnswer" passages, the "temporal" field must be an empty list.
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- “docid”: must remain the same as provided.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality positive passages, prioritizing quality over quantity.

5. "negative_passages": follow the same format as "positive_passages", but represent contexts that are temporally mismatched or semantically irrelevant to the query. Based on the query's temporal expressions, you must generate "negative_passages" that cover all "TemporalQueryType", including "Explicit", "Implicit", and "TemporalAnswer", when possible. They must be hard, temporally-confused, yet diverse in Allen relations questions that cover all following cases:
- Case 1: Questions with temporal expressions that mismatch with the query’s temporal expression(s): 
    - If the query specifies a span (e.g., 2005–2007), use an interval that falls completely outside it (e.g., 2008, before 2005, after 2007).
    - Adjacent years or ranges are valid negatives (e.g., query = 2010, negative = 2009).
    - Use shifted but non-overlapping intervals (e.g., query = 2010, negative = 2012–2014).
    - Include misleading implicit cues (e.g., “shortly after 2011” vs. query “in 2010”).
    - You may reuse the same passage from "positive_passages" but replace its temporal expressions with mismatched ones.
- Case 2: Questions with same temporal expression but irrelevant event/entity
    - Questions with overlapping or identical temporal expressions but targeting a different subject.
    - Example: query = “Ronaldo’s career in 2010” vs. negative = “Messi’s career in 2010”.
    - This ensures negatives are temporally aligned but semantically irrelevant.
- Case 3: "TemporalAnswer"-type questions that are either:
    - Irrelevant to the query.
    - Or ask for non-existent temporal information in the query context.
    - Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality negative passages, prioritizing quality over quantity.  
- Do not create trivial negatives (e.g., completely unrelated random text).
- Ensure no accidental overlap with valid facts in the document.

6. "temporal_query_type": Must be "Explicit", "Implicit", or "TemporalAnswer". If the temporal contains a clear date/month/year, it is "Explicit", NOT "Implicit". If the passage asks for which date/month/year of an event, it is "TemporalAnswer".
   
7. Final output: Only output valid JSON(s). Do not explain, add comments, or include extra text, since your output will be parsed automatically.

### Demonstration 1
Input:
docid: 2465
query_id: 0
query: "On 2 July 2010 , after helping Setúbal avoid top-flight relegation , Barbosa was released by Porto , signing a three-year contract with S.C . Braga."
SUTIME's output: [{'timex-value': '2010-07-02', 'start': 3, 'end': 14, 'text': '2 July 2010', 'type': 'DATE', 'value': '2010-07-02'}, {'timex-value': 'P3Y', 'start': 111, 'end': 121, 'text': 'three-year', 'type': 'DURATION', 'value': 'P3Y'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?.
Example negative passages: "Hélder Barbosa played for which team from 2002 to 2009?", "Hélder Barbosa played for which team from 2006 to 2009?".

Output:
{"query_id":0,"query":"On 2 July 2010, after helping Setúbal avoid top-flight relegation, Barbosa was released by Porto, signing a three-year contract with S.C. Braga.","temporal":["2 July 2010","three-year contract"],"positive_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2010 to 2013?","temporal":["from 2010 to 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for on 2 July 2010?","temporal":["2 July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa join after leaving Porto in July 2010?","temporal":["after leaving Porto in July 2010"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa sign a three-year contract with?","temporal":["three-year contract"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"After being released by Porto, which team did Hélder Barbosa sign a contract with?","temporal":["After being released by Porto"],"allen_relation":"MetBy","temporal_query_type":"Implicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between July 2010 and July 2013?","temporal":["between July 2010 and July 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"When did Hélder Barbosa sign a contract with S.C. Braga.?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2465,"text":"How long did Hélder Barbosa's contract with S.C. Braga last?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2002 to 2009?","temporal":["from 2002 to 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between 2006 and 2009?","temporal":["between 2006 and 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for after 2014?","temporal":["after 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for during 2008?","temporal":["during 2008"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for prior to 2010?","temporal":["prior to 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Ronaldo play for in July 2010?","temporal":["in July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 1

### Demonstration 2
Input:
docid: 2466
query_id: 1
query: "Rarely used in the first months , he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish ."
SUTIME's output: [{'timex-value': 'PXM', 'start': 15, 'end': 31, 'text': 'the first months', 'type': 'DURATION', 'value': 'PXM'}, {'timex-value': '2011-01', 'start': 78, 'end': 90, 'text': 'January 2011', 'type': 'DATE', 'value': '2011-01'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?"
Example negative passages: Hélder Barbosa played for which team from 2002 to 2009?; Hélder Barbosa played for which team from 2006 to 2009?.

{"query_id":1,"query":"Rarely used in the first months, he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish .","temporal":["after the January 2011 departure of Matheus"],"positive_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for between January 2011 and December 2011?","temporal":["between January 2011 and December 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for as of January 2011?","temporal":["as of January 2011"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after Matheus departed in January 2011?","temporal":["after Matheus departed in January 2011"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa contribute goals to during the 2011 season?","temporal":["during the 2011 season"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for throughout 2011?","temporal":["throughout 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"When did Hélder Barbosa start getting more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2466,"text":"When did Hélder Barbosa begin gaining more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for before 2010?","temporal":["before 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after 2013?","temporal":["after 2013"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for in 2007?","temporal":["in 2007"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for prior to joining Braga?","temporal":["prior to joining Braga"],"allen_relation":"Before","temporal_query_type":"Implicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for during 2014?","temporal":["during 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"After the January 2011 departure of Matheus, did Ronaldo get more playing time?","temporal":["After the January 2011 departure of Matheus"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 2"""

import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")

def normalize_and_split_string_by_punctuation(text):    
    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)
    
    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)    

    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"
    
    # Remove : and ;
    punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    return [x.strip() for x in re.findall(punct_regex, text)]



In [ ]:
a = {"query_id":2,"query":"According to some sources , Disney worried about the rising criminality of the city . A neighboring family had two adolescent children involved in a car barn robbery , and Disney feared that crime would taint his own children . In 1906 he moved with his family to a farm near Marceline , Missouri . Disney and his family settled there in April , 1906 . On March 5 , he bought a farm . Its previous owner William E . Crane had died in November , 1905 . Crane was a veteran of the American Civil War and his house predated the foundation of Marceline . He bought the farm for $3,000 or $75 per acre . On April 3 , Disney bought an adjoining tract of about from Cranes widow . He paid an additional $450 .","positive_passages":[{"docid":2756,"text":"What was the residence of Elias Disney from 1906 to 1910?"},{"docid":2756,"text":"What was the residence of Elias Disney in October 1906?"}],"negative_passages":[{"docid":2756,"text":"What was the residence of Elias Disney from 1911 to 1912?"},{"docid":2756,"text":"What was the residence of Elias Disney from 1911 to 1913?"}]}
text = a["query"]
query_id = a["query_id"]
docid = a["positive_passages"][0]["docid"]
positive_passages = a["positive_passages"]
negative_passages = a["negative_passages"]

positive_passages = [x['text'] for x in positive_passages]
negative_passages = [x['text'] for x in negative_passages]

query_list = normalize_and_split_string_by_punctuation(text)
final_query_list = []
final_sutime_list = []

messages = []

final_query_list = []
final_sutime_list = []

buffer = []

for q in query_list:
    parsed = sutime.parse(q)

    if len(parsed) == 0:
        # No temporal expression → keep merging
        buffer.append(q)
    else:
        # Check if merging with buffer creates more temporal expressions
        if buffer:
            merged = " ".join(buffer + [q])
            if len(sutime.parse(merged)) > 1:
                temp = " ".join(buffer)
                final_query_list.append(temp)
                final_sutime_list.append(sutime.parse(temp))
                buffer = [q]
                continue
        buffer.append(q)

# Flush leftover
if buffer:
    merged = " ".join(buffer)
    temp = sutime.parse(merged)
    if len(temp) == 1:
        final_query_list.append(merged)
        final_sutime_list.append(temp)

for q, sutime_output in zip(final_query_list, final_sutime_list):
    # sutime_output = sutime.parse(q.lower()) 
    
    if not (len(sutime_output) > 0 and len(word_count_regex.findall(q)) > 5):
        continue

    for t in sutime_output:
        s, e = t["start"], t["end"]
        t["value"] = q[s:e]

    content = f"Input:\ndocid: {docid}\nquery_id: {query_id}\nquery: {q}\nSUTIME's output: {sutime_output}\nExample positive passages: {','.join(positive_passages)}\nExample negative passages:{';'.join(negative_passages)}\nOutput:"
    
    messages.append([
        {"role":"system", "content":prompt_template},
        {"role":"user", "content":{content},}
    ])

In [ ]:
print(query_list)
print(final_query_list, len(final_sutime_list), [len(sutime.parse(x.lower())) for x in final_query_list])

In [ ]:
output = llm.chat(
    messages=messages, 
    sampling_params=sampling_params,
    # chat_template_kwargs={"enable_thinking": True},
)

In [ ]:
output

In [ ]:
temporal_answer_list = [
    # Point in time
    "what day",
    "what time",
    "what date",
    "what month",
    "what year",
    "which day",
    "which time",
    "which date",
    "which month",
    "which year",
    "when did",
    "when was",
    "when were",
    "at what time",
    "at what date",
    "at what year",
    "on what day",
    "on what date",
    "on what year",

    # Duration / span
    "how long",
    "how many days",
    "how many weeks",
    "how many months",
    "how many years",
    "for how long",
    "over what period",
    "during what years",
    "during which year",
    "for what duration",
    "in what year range",
    "between what years",

    # Frequency / recurrence
    "how often",
    "how frequently",

    # Relative temporals
    "since when",
    "until when",
    "from when",
    "from what year",
    "from what date",
    "to what year",
    "to what date",
    "up to when",
    "as of when",
    "by what year",
    "by when",
    "around when",
    "at what age",
]


def fix_implicit_temporal(passage, sutime):
    """
    Adjust temporal_query_type if 'implicit' but tagger detects explicit datetime.
    """
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Implicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    if len(sutime.parse(passage["temporal"][0].lower())) > 0:
        passage["temporal_query_type"] = TemporalQueryType.Explicit
        
    lower_passage = passage["text"].lower()
    
    # Make sure that the
    implicit_temporal = []
    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            implicit_temporal.append(passage["text"][start:start+len(t)])    
    
    if not implicit_temporal:
        return {}
    
    passage["temporal"] = implicit_temporal
    return passage


def fix_temporal_answer(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.TemporalAnswer:
        return passage

    lower_passage = passage['text'].lower()
    if any(word in lower_passage for word in temporal_answer_list):
        for word in temporal_answer_list:
            if word in lower_passage:
                lower_passage = lower_passage.replace(word, "")
        
    # Ensure that it passes the SUTIME tagger.
    if len(sutime.parse(lower_passage)) > 0:
        return {}
    else:
        passage['temporal'] = []
        passage['allen_relation'] = AllenRelation.Empty
    # print(passage)
    return passage


def fix_explicit_temporal(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Explicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    lower_passage = passage["text"].lower()
    temporal = []

    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            temporal.append(passage["text"][start:start+len(t)])

    if not temporal:
        return {}

    passage["temporal"] = temporal
    return passage


def validate_passages(passages, sutime):
    """
    Validate and filter passages.
    """
    valid = []
    
    for p in passages:
        p = fix_implicit_temporal(p, sutime)
        p = fix_explicit_temporal(p, sutime)
        p = fix_temporal_answer(p, sutime)
        
        if p:
            valid.append(p)
    return valid


def validate_temporal(js, max_temporal_expressions=3):
    lower_query = js["query"].lower()
    temporal = []
    
    # Ensure that the temporal is extracted as written from the original query
    for t in js["temporal"]:
        start = lower_query.find(t.lower())
        if start != -1:
            temporal.append(js["query"][start:start+len(t)])

    if len(temporal) == 0 or len(temporal) > max_temporal_expressions:
        # print(temporal)
        return []
    return temporal


temporal_jsonl = []
temp = []
for sample in output:
    sample = sample.outputs[0].text # Get the generated text
    list_of_raw_js = []
    query_set = set()
    
    for x in sample.split("\n"): # Split by newline to get multiple JSONs if any
        parts = x.split(',{"query_id"')
        if len(parts) > 0:
            json_strings = [parts[0]] + [',{"query_id"' + p for p in parts[1:]]
            list_of_raw_js.extend(json_strings)
        else:
            list_of_raw_js.append(x)
    
    for raw_js in list_of_raw_js:
        raw_js = raw_js.strip()
        if raw_js == "":
            continue
        try:
            js = TemporalAnnotation.model_validate_json(repair_json(raw_js, ensure_ascii=False)).model_dump()
            
            if len(js["temporal"]) == 0 or len(js["positive_passages"]) == 0 or len(js["negative_passages"]) == 0 or js["query"] in query_set:
                continue
            else:
                print(js["query"])
                js["temporal"] = validate_temporal(js)
                
                if len(js["temporal"]) == 0:
                    continue
                
                # Validate passages
                js["positive_passages"] = validate_passages(
                    js["positive_passages"], sutime
                )
                
                if len(js["positive_passages"]) == 0:
                    continue
                
                js["negative_passages"] = validate_passages(
                    js["negative_passages"], sutime
                )
                
                if len(js["negative_passages"]) == 0:
                    continue
                
                temp.append(js)
                query_set.add(js["query"])
                # positive = defaultdict(int)
                # negative = defaultdict(int)
                # allen_relation = defaultdict(int)

                # for j in js["positive_passages"]:
                #     positive[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                # for j in js["negative_passages"]:
                #     negative[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                
                # print(positive)
                # pprint(positive)
                # print(positive)
                # pprint(negative)

        except Exception as e:
            print("Error:", e)
            print("Offending JSON:", raw_js)
            continue
temporal_jsonl.extend(temp)

In [ ]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v3/test.jsonl", temporal_jsonl, jsonl=True)

In [ ]:
del llm
cleanup_dist_env_and_memory()

## Post-processing

In [ ]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

In [ ]:
# temporal_jsonl
positive = defaultdict(int)
negative = defaultdict(int)
allen_relation = defaultdict(int)
for x in temp_jsonl:
    for i in x["positive_passages"]:
        positive[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
    for i in x["negative_passages"]:
        negative[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
pprint(positive)
pprint(negative)

### Check for relative temporal expressions in the queries

In [ ]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

# new_temp_jsonl = []

count = 0

for x in temp_jsonl:

    if "now" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "now")
        count += 1
        # break
    if "today" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "today")
        count += 1
        # break
    if "current" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "current")
        count += 1 
        # break
    if "yesterday" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "yesterday")
        count += 1     
        # break
        
    # if count != 0 and (any([i["temporal_query_type"] == "Explicit" for i in x["positive_passages"]]) or any([i["temporal_query_type"] == "Explicit" for i in x["negative_passages"]])):
    #     continue
    
#     new_temp_jsonl.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)

### Check for limited positive/negative passages

In [ ]:
# new_temp_jsonl = []
for x in temp_jsonl:
    if len(x["positive_passages"]) <= 1:
        print("pos", x["query_id"])
        count += 1
        continue
    if len(x["negative_passages"]) <= 1:
        print("nega", x["query_id"])
        count += 1
        continue
#     new_temp_jsonl.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)

### Check for positive/negative passages that only contain a reference year (2025)

In [ ]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

count = 0
for x in temp_jsonl:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if all(t):
        print(x["query_id"])
        print(t)
        count += 1
print(count)   

In [68]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal.jsonl", jsonl=True)

Reading the json file...
The file is of type: <class 'list'>
The file contains 11693 items.


In [69]:
count = 0
count2 = 0
for x in temp:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if any(t):
        x["positive_passages"] = [j for j in x["positive_passages"] if "2025" not in j["text"]]

In [70]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal_v2.jsonl", temp, jsonl=True)

The file contains 11693 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal_v2.jsonl


### Ensure that either positive and negative passages must contain at least one explicit/implicit temporal

In [ ]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

count_pos = 0
count_neg = 0
count_both = 0
count = 0
query_set = set()
final_query_list = []

for x in temp_jsonl:
    check_pos = False

    for i in x["positive_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_pos = True
            break 

    if check_pos == False:
        # print("pos", x["query_id"], x["query"][:10])
        count_pos += 1
        query_set.add(x["query_id"])

    check_neg = False    
    for i in x["negative_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_neg = True
            break

    if check_neg == False:
        count_neg += 1
        # print("neg", x["query_id"], x["query"][:10])
        query_set.add(x["query_id"])
        
    # if check_neg == True and check_pos == True:
    #     count += 1
    #     count_both += 1
    #     query_set.add(x["query_id"])
    #     final_query_list.append(x)
    
    if not check_neg and not check_pos:
        count_both += 1
        print(x["query_id"], x["query"][:10])

# print(count)

In [ ]:
print(count, count_pos, count_neg, count_both)

# Time-sensitive QA

In [3]:
from wikimapper import WikiMapper

# wiki_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/enwiki-20211220-pages-articles.jsonl", jsonl=True)
mapper = WikiMapper("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/index_enwiki-20220820.db")

easy_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/train.easy.json", jsonl=True)

easy_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/dev.easy.json", jsonl=True)

easy_test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/test.easy.json", jsonl=True)

hard_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/train.hard.json", jsonl=True)

hard_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/dev.hard.json", jsonl=True)

hard_test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/test.hard.json", jsonl=True)

train_jsonl = easy_train_jsonl + hard_train_jsonl
dev_jsonl = easy_dev_jsonl + hard_dev_jsonl
test_jsonl = easy_test_jsonl + hard_test_jsonl
# print(mapper.title_to_id("Carl_Eric_Almgren")) # Q5040099
# print(mapper.url_to_id("/wiki/Carl_Eric_Almgren#P39#1")) # None

## Create the corpus

### Split the large JSONL into chunk size JSONLs

In [ ]:
corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/temp.parquet")

# def fix_title(x):
#     return "_".join(x[1:-1].split(" "))


# if __name__ == "__main__":
#     with Pool(16) as pool:  # use all available cores
#         corpus_title = list(
#             tqdm(
#                 pool.imap(fix_title, corpus_title),
#                 total=corpus_length,
#             )
#         )

# corpus_parquet["original_title"] = corpus_parquet["title"]
# corpus_parquet["title"] = corpus_title
# corpus_parquet.to_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/temp.parquet"
# )

In [ ]:
current_corpus = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.jsonl", jsonl=True)

In [ ]:
import random

def sample_excluding(large_list, small_list, target_size=100_000):
    # Convert small_list to a set for O(1) membership checks
    small_set = set(small_list)

    # Filter out any text that appears in small_list
    filtered = [x for x in large_list if x not in small_set]

    if len(filtered) < target_size:
        raise ValueError(f"Not enough unique items ({len(filtered)}) to sample {target_size} items.")

    # Sample without replacement
    return random.sample(filtered, target_size)

small_list = [x["text"] for x in current_corpus]
large_list = corpus_parquet["text"].sample(1000000).tolist()

In [ ]:
result = sample_excluding(large_list, small_list, 100000-23009)

In [ ]:
result_jsonl = [
    {
        "docid": ix,
        "text": x
    }
    for ix, x in zip(range(23009, 100_000), result)
]

In [ ]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/extra_corpus.jsonl", result_jsonl, jsonl=True)

In [ ]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus.jsonl", jsonl=True)

In [ ]:
# def func(x):
#     return mapper.title_to_id(x)

# corpus_title = corpus_parquet["title"].tolist()
# corpus_length = len(corpus_title)

# if __name__ == "__main__":
#     with Pool(16) as pool:  # use all available cores
#         corpus_id = list(
#             tqdm(
#                 pool.imap(func, corpus_title),
#                 total=corpus_length,
#             )
#         )
# corpus_parquet["id"] = corpus_id
# corpus_parquet.to_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/temp.parquet"
# )

In [ ]:
"at Comenius University Bratislava.  Section::::Biography. Born 6 December 1938 in Prague, studied physics and mathematics at Comenius University (CU). In 1964 he obtained his PhD in statistics"

## Extract the wiki articles

3019
3020
3021
3022
'Royal_Carillon_School_"Jef_Denyn"' Q2279602

3743
3744
3745
3746
3747
3748
3749
3750
Escadron_de_Chasse_1/2_Cigognes Q3057701

6566
6567
6568
Parti_communiste_du_Québec_(sovereigntist)

8338
8339
8340
International_Association_for_Political_Science_Students

9691
9692
9693
RE/MAX_Field Q7697921

12889
12890
12891
12892
12893
12894
Darren_Hughes_(footballer)

13308
13309
13310
Maicon_Pereira_de_Oliveira

17414
17415
17416
17417
'Royal_Carillon_School_"Jef_Denyn"' Q2279602

18152
18153
18154
18155
18156
18157
18158
18159
Escadron_de_Chasse_1/2_Cigognes Q3057701

21048
21049
21050
Parti_communiste_du_Québec_(sovereigntist)

22867
22868
22869
International_Association_for_Political_Science_Students

24256
24257
24258
RE/MAX_Field Q7697921

27536
27537
27538
27539
27540
27541
Darren_Hughes_(footballer)

27963
27964
27965
Maicon_Pereira_de_Oliveira

In [ ]:
wiki_url_regex = re.compile(r"([^#]+)")

def process(x):
    return mapper.url_to_id(re.sub(r"amp;", "", wiki_url_regex.match(x['idx'])[0]))

train_id = []
dev_id = []
test_id = []

if __name__ == '__main__':
    with Pool(16) as pool:  # use all available cores
        train_id = list(tqdm(
            pool.imap(process, train_jsonl), 
            total=len(train_jsonl)
        ))

    with Pool(16) as pool:  # use all available cores
        dev_id = list(tqdm(
            pool.imap(process, dev_jsonl), 
            total=len(dev_jsonl)
        ))

    with Pool(16) as pool:  # use all available cores
        test_id = list(tqdm(
            pool.imap(process, test_jsonl), 
            total=len(test_jsonl)
        ))

for i in range(3019, 3022 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q2279602"
    train_id[i] = "Q2279602"

for i in range(3743, 3750 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q3057701"
    train_id[i] = "Q3057701"

for i in range(9691, 9693 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q7697921"
    train_id[i] = "Q7697921"

for i in range(17414, 17417 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q2279602"
    train_id[i] = "Q2279602"

for i in range(18152, 18159 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q3057701"
    train_id[i] = "Q3057701"

for i in range(24256, 24258 + 1):
    print(train_jsonl[i]["idx"])
    train_jsonl[i]["wiki_id"] = "Q7697921"
    train_id[i] = "Q7697921"

### Filter out hard set documents from the wikipedia corpus (22453 entries in total)

In [ ]:
result_df = pd.DataFrame()
json_id_set = set(train_id + dev_id + test_id)

temp_df = corpus_parquet[
    corpus_parquet["id"].isin(json_id_set) & corpus_parquet["id"].notna()
]

result_df = pd.concat([result_df, temp_df])
print(len(result_df))

result_df = result_df.reset_index()
result_df.to_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
subset_corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)

### Save the splits into parquets for later use

In [ ]:
for iy, (i, j) in tqdm(enumerate(zip(test_jsonl, test_id))):
    temp = subset_corpus_parquet[subset_corpus_parquet["id"].values == j]
    if len(temp) == 0:
        print(iy)
    # i["docid"] = []
    i["wiki_id"] = temp["id"].values[0] if len(temp["id"].values) != 0 else None
    answer = i["targets"]

    # for a in answer:
    #     for ix, t in enumerate(temp["text"]):
    #         # Strip all punctuations for more correct search
    #         t = re.sub(r'[^\w\s]','',t, re.UNICODE)
    #         if a in t:
    #             i["docid"].append(temp.index[ix])
df = pd.DataFrame(test_jsonl)
df["unanswerable"] = df["targets"].apply(lambda x: x == [''])
df = df.dropna()
df.to_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet"
)

In [ ]:
# df_train = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.parquet")
# df_dev = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/dev.parquet")
# df_test = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet")

# df_train = df_train.dropna()
# df_dev = df_dev.dropna()
# df_test = df_test.dropna()

# df_train.to_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.parquet"
# )
# df_dev.to_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/dev.parquet"
# )
# df_test.to_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet"
# )
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
df_train = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.parquet"
)
df_dev = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/dev.parquet"
)
df_test = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet"
)

print(len(df_train[df_train["wiki_id"].isna()]), len(df_train[df_train["unanswerable"].values == True]))
print(len(df_dev[df_dev["wiki_id"].isna()]), len(df_dev[df_dev["unanswerable"].values == True]))
print(len(df_test[df_test["wiki_id"].isna()]), len(df_test[df_test["unanswerable"].values == True]))

### Extract the paragraphs for each question in the train split

In [ ]:
def normalize_and_split_string_by_punctuation(text, add_title=False):
    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)

    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)

    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"

    # Remove : and ;
    punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"

    if add_title:
        title = text.split(": ")[0]
        return [
            f"{title}: " + x.strip()
            for ix, x in enumerate(re.findall(punct_regex, text))
            if ix != 0
        ]

    return [x.strip() for x in re.findall(punct_regex, text)]


def search_paragraphs_index(recon_start_end_list, answer_start_end, recon_context, targets, paragraphs):
    """Return the index of the paragraph span that contains the answer span."""
    potential_ix = []
    for ix, (s, e) in enumerate(recon_start_end_list):
        if targets in paragraphs[ix]['text']:
            potential_ix.append(ix)
        if answer_start_end[0] >= s and answer_start_end[1] <= e and targets in paragraphs[ix]['text']:
            return ix
    
    potential_start_end_list = [recon_start_end_list[x] for x in potential_ix]
    for ix, (s, e) in enumerate(potential_start_end_list):
        if answer_start_end[0] >= s:
            return ix
    return -1

def format_paragraph(paragraphs, art_title, idx, dedup_titles):
    par_title = paragraphs[idx]["title"]
    if dedup_titles and art_title == par_title:
        title_str = art_title
    else:
        title_str = f"{art_title} - {par_title}"
    return f"{title_str}: {paragraphs[idx]['text']}"

def extract_paragraphs_with_answers(paragraphs, answer_starts, answer_ends, targets, dedup_titles=True):
    """
    Given Wikipedia-style paragraphs and answer spans (start/end indices),
    return a list of disambiguated paragraph strings that include both
    article title and section title.
    
    Parameters
    ----------
    paragraphs : list[dict]
        Each dict must have keys: "title", "text".
    answer_starts : list[int]
        List of answer start indices (aligned with reconstructed context).
    answer_ends : list[int]
        List of answer end indices.
    dedup_titles : bool, default=True
        If True, collapses "Title - Title" into "Title".
    
    Returns
    -------
    list[str] : formatted paragraphs with titles and text
    """
    
    title_set = set()
    start_end_list = []
    recon_context = ""
    final_paragraph_list = []

    # Build context + paragraph spans
    for par in paragraphs:
        title = par["title"]
        text = par["text"].strip()

        # Title case
        if len(title_set) == 0: 
            recon_context += title.strip()
            title_set.add(title)

        # Add article/section title once
        if title not in title_set:
            recon_context += " " + title.strip() + " . "
            title_set.add(title)

        start_idx = len(recon_context)
        recon_context += " " + text.strip() + " "
        end_idx = len(recon_context)

        start_end_list.append((start_idx, end_idx))
    recon_context = recon_context.replace("  ", " ")

    # Match answers back to paragraph(s)
    for t, s, e in zip(targets, answer_starts, answer_ends):
        ix = search_paragraphs_index(start_end_list, (s, e), recon_context, t, paragraphs)
        art_title = paragraphs[0]["title"]

        if ix != -1:
            final_paragraph_list.append(format_paragraph(paragraphs, art_title, ix, dedup_titles))
            continue

        # fallback: search in recon_context
        t_lower = t.lower()
        for ix, (_, end) in enumerate(start_end_list):
            if t_lower in recon_context[:end].lower():
                nearby_idxs = range(max(0, ix - 1), min(len(paragraphs), ix + 1))
                combined = " ".join(
                    format_paragraph(paragraphs, art_title, j, dedup_titles)
                    for j in nearby_idxs
                )
                final_paragraph_list.append(combined)
                break

    return final_paragraph_list

In [ ]:
output_jsonl = []
count = 0 

for i in range(len(df_train)):
    sample = df_train.iloc[i]
    
    if sample["unanswerable"]:
        continue
    
    targets = sample['targets'].tolist()
    start, end = sample["from"].tolist(), sample["end"].tolist()
    paragraphs = sample['paragraphs'].tolist()
    # print(len(sample['context']), sample['context'])

    result = extract_paragraphs_with_answers(paragraphs, start, end, targets)
    
    if len(result) == 0:
        # Final resort
        temp = normalize_and_split_string_by_punctuation(sample["context"])
        for ix, chunk in enumerate(temp):
            if targets[0] in chunk:
                start = max(0, ix - 5)
                end = min(len(temp), ix + 5)
                result = temp[start:end]
                break
        if len(result) == 0:
            print(i, sample['targets'][0], sample['targets'][0] in sample['context'])
            count += 1
            
    for res in result:
        output_jsonl.append({
            "query_id": i,
            "query": res,
            "positive_passages": [{"docid":sample["wiki_id"], "text": sample["question"]}]
        })

In [ ]:
# new_value = df_train.iloc[13448]["paragraphs"].tolist()
# new_value.pop(15)
# new_value[14] = {
#     "text": "Section:::: U.S . House of Representatives. Soto won the Democratic nomination to succeed Representative Alan Grayson , who had stepped down to run in the primary for U.S . Senate in Floridas 9th congressional district . Soto earned 36% of the vote in a four-way primary election . The district is Democratic-leaning and contains all of Osceola County and parts of Orange and Polk counties . Soto has represented the majority of this district while serving in the Florida House of Representatives and the Florida Senate . The Orlando Sentinel endorsed him in his primary race , calling him an effective lawmaker . Soto won the general election for the seat , defeating Republican Wayne Liebnitzky , 57–43% .'",
#     "title": "Elections",
# }
# df_train["paragraphs"].iat[13448] = np.array(new_value)
# df_train["paragraphs"].iat[28090] = np.array(new_value)

# new_value = df_train.iloc[9676]["paragraphs"]
# new_value[-1]["text"] = 'While he slowly recovered from a car crash that occurred while vacationing in France in 1958, Jack returned to the studio and made sure his name was featured in studio press releases. From 1961 to 1963, the studio\'s annual net profit was a little over $7\xa0million. Warner paid an unprecedented $5.5\xa0million for the film rights to the Broadway musical "My Fair Lady" in February 1962. The previous owner, CBS director William S. Paley, set terms including half the distributor\'s gross profits "plus ownership of the negative at the end of the contract."'
# new_value = np.append(
#     new_value,
#     {
#         "text": 'In 1963, the studio\'s net profit dropped to $3.7\xa0million. By the mid-1960s, motion picture production was in decline, as the industry was in the midst of a painful transition from the Golden Age of Hollywood to the era now known as New Hollywood. Few studio films were made in favor of co-productions (for which Warner provided facilities, money and distribution), and pickups of independent pictures.  With the success of the studio\'s 1964 film of Broadway play "My Fair Lady", as well as its soundtrack, Warner Bros. Records became a profitable subsidiary. The 1966 film "Who\'s Afraid Of Virginia Woolf?" was a huge success.',
#         "title": "New owners",
#     },
# )
# new_value = np.append(
#     new_value,
#     {
#         "text": "In November 1966, Jack gave in to advancing age and changing times, selling control of the studio and music business to Seven Arts Productions, run by Canadian investors Elliot and Kenneth Hyman, for $32\xa0million. The company, including the studio, was renamed Warner Bros.-Seven Arts.",
#         "title": "New owners",
#     },
# )
# df_train["paragraphs"].iat[9676] = new_value
# df_train["paragraphs"].iat[24226] = new_value

# new_value = df_train.iloc[8837]["paragraphs"]
# new_value[3]["text"] = "Section::::State legislature and U.S . Congress. Milledge's political career began in 1779, when he was elected to the patriot general assembly. After serving as the attorney general of Georgia, Milledge was a member of the Georgia General Assembly. While in the General Assembly, he spoke out forcefully against the Yazoo Land Acts. In 1792, the House of Representatives declared the seat of Anthony Wayne vacant due to disputes over his residency. Milledge was elected to the Second Congress to fill this vacancy and served from November 22, 1792, to March 3, 1793."
# new_value[4]["text"] = "Later, Milledge would be elected to the Fourth and Fifth Congresses , serving from March 4 , 1795 to March 3 , 1799 . In 1801 , he was again elected to Congress , this time as a Democratic-Republican , and served from March 4 , 1801 , until he resigned in May 1802 to become Governor of Georgia .'"
# df_train["paragraphs"].iat[8837] = new_value
# df_train["paragraphs"].iat[23365] = new_value

# df_train["paragraphs"].iat[7655] = np.insert(
#     df_train.iloc[7655]["paragraphs"],
#     -5,
#     {
#         "text": "In July 2010 Pagano was signed by A.S . Livorno Calcio.",
#         "title": "Livorno",
#     },
# )
# df_train["paragraphs"].iat[22156] = np.insert(
#     df_train.iloc[22156]["paragraphs"],
#     -5,
#     {
#         "text": "In July 2010 Pagano was signed by A.S . Livorno Calcio.",
#         "title": "Livorno",
#     },
# )

### Merge duplicate passages together and accumulate the questions of the same passage

In [ ]:
output_jsonl[0]

In [ ]:
# temp = defaultdict(set)
# new_train_jsonl = []

# for i in output_jsonl:
#     temp[i["query"]].add(
#         (i["positive_passages"][0]["docid"], i["positive_passages"][0]["text"])
#     )

# for k, v in temp.items():
#     temp[k] = [{"docid":x[0], "text":x[1]} for x in v]

# for ix, (k, v) in tqdm(enumerate(temp.items())):
#     try:
#         if list(v)[0]["docid"] is None:
#             continue
#         new_train_jsonl.append({
#             "query_id": ix,
#             "query": k,
#             "positive_passages": [x for x in v],
#         })
#     except Exception as e:
#         print(v)
write_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl",
    new_train_jsonl,
    jsonl=True,
)

## VLLM generation

In [ ]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]

print(TemporalAnnotation.model_json_schema())

In [ ]:
prompt_template = """You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME's output for your reference. We also provide you with previously annotated positive samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.
Your final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.
Your temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. 
You must follow the definitions and instructions below.

* Allen relations and descriptions:
- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?
- After: Did ‘Event A’ occur after ‘Event B’ without any overlap between the two events?
- Meets: Did ‘Event A’ end in the same time as ‘Event B’ began? Answer True or False.
- MetBy: Did ‘Event B’ end in the same time as ‘Event A’ began? Answer True or False.
- Overlaps: Did ‘Event A’ begin before ‘Event B’ and end before ‘Event B’ ended, with some overlap between the two events?
- OverlappedBy: Did ‘Event B’ begin before ‘Event A’ and end before ‘Event A’ ended, with some overlap between the two events?
- Starts: ‘Event A’ begin in the same time as ‘Event B’, but end before ‘Event B’ ended?
- StartedBy: Did ‘Event B’ begin in the same time as ‘Event A’, but end before ‘Event A’ ended?
- During: Did ‘Event A’ begin after ‘Event B’ began and end before ‘Event B’ ended, being entirely contained within ‘Event B’?
- Contains: Did ‘Event A’ begin before ‘Event B’ began and end after ‘Event B’ ended, entirely containing ‘Event B’?
- Finishes: Did ‘Event A’ begin after ‘Event B’ began and end in the same time as ‘Event B’?
- FinishedBy: Did ‘Event B’ begin after ‘Event A’ began and end in the same time as ‘Event A’?
- Equals: Did ‘Event A’ begin in the same time as ‘Event B’ and end in the same time as ‘Event B’?
- Empty: a special case for TemporalAnswer where there is no temporal expression.

* Temporal signals (examples of what can appear in queries or passages): before, prior to, until, after, following, since, in, on, as of, for duration, during, while, when, from...to..., between, by, up to, first, last, around, as soon as, as long as, for, over, all through, throughout, etc.

* Taxonomy of "positive_passages" and "negative_passages" with examples that you should generate:
- Explicit temporal constraints (i.e., always have clear temporal expressions that can be anchored to a specific datetime): "who won the state of Texas in 2008?"; "what kind of government does Iran have after 1979?".
- Implicit temporal constraints (i.e., events that cannot be anchored to specific datetime. The rule of thumb is: if there is date or month or year in the question, like "before the **2004** general election", it is not implicit temporal): "who was the president after JFK died?"; "what team did Michael Jordan play for after the Bulls?".
- TemporalAnswer (i.e., Questions that inquire the datetime of an event instead of a general question with temporal constraints, usually starts with "when" or "what/which + day/date/month/year". There is no temporal expression in it.): "what year did the Knicks win the championship?"; "when was the United Nations founded?".

* Instructions:
1. You must output one or multiple valid JSONs, delimited by a newline, strictly following this pydantic schema:
{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'positive_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Positive Passages', 'type': 'array'}, 'negative_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Negative Passages', 'type': 'array'}}, 'required': ['query_id', 'query', 'temporal', 'positive_passages', 'negative_passages'], 'title': 'TemporalAnnotation', 'type': 'object'}.

2. "query_id": must be assigned with the provided "query_id".

3. “query":
- A query is a provided sentence that may contain one or many temporal expressions.
- Please note that SUTIME only provides explicit temporal expressions for the query and they are not perfect; therefore, you must double-check them and further detect additional explicit and implicit temporal expressions such as events.
- You must extract all events that are anchored to specific datetime (e.g., “2000 FA Cup Final”, “the 2007 election”, etc.).
- The "temporal" field should be extracted as written from the query's text and they must be concise, such as: "in April, 1906", "2 July 2010", "2004 general election", etc.

4. "positive_passages":
- Natural, QA-style questions with diverse phrasing that seek information from the query. Each question must contain exactly one extractable temporal expression, logically align with one of the query's temporal expressions, yet be diverse in Allen relations.
- Based on the query's temporal expressions, you must generate "positive_passages" that cover all "TemporalQueryType", including 'Explicit', 'Implicit', and 'TemporalAnswer', when possible:
    - You MUST prioritise generating questions with explicit temporal constraints, like "in 2010", "from 2010 to 2015", etc. They must logically align with the query’s temporal expression(s), such as being equals, overlapping with, or being contained within the query’s temporal expression(s). If the query has multiple temporal expressions, the generated questions must logically align with at least one of them.
    - You should also prioritise generating questions with implicit temporal constraints, such as "after EVENT", "before EVENT". These must logically align with the query’s temporal expression(s).
    - For both explicit and implicit "TemporalQueryType" questions, you must not use phrases like what/which date/day/month/year/time or when etc., that inquire about time. You must not confuse this with TemporalAnswer questions.
    - You can also generate "TemporalAnswer" (asking for datetime/duration/time-range of an EVENT, e.g., "When did EVENT happen?", "What time did he arrive?"). For this type of question, you must not add any temporal expression and you must set: "TemporalQueryType": "TemporalAnswer", "allen_relation": "Empty", and "temporal": []. Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- "temporal" field: must extract all exact text spans of the temporal expressions that exist in the generated question (not normalized or paraphrased). Regarding explicit and implicit "TemporalQueryType", they must be concise temporal expressions as written, such as "after July 2010", "from 2012 to 2014", that are not just normalized dates. For instance, instead of "In 1906 he moved with his family to a farm", prefer the concise "In 1906" and keep prepositions or context words that anchor the time, e.g., 'from', 'in', 'after', etc. Regarding "TemporalAnswer" passages, the "temporal" field must be an empty list.
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- “docid”: must remain the same as provided.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality positive passages, prioritizing quality over quantity.

5. "negative_passages": follow the same format as "positive_passages", but represent contexts that are temporally mismatched or semantically irrelevant to the query. Based on the query's temporal expressions, you must generate "negative_passages" that cover all "TemporalQueryType", including "Explicit", "Implicit", and "TemporalAnswer", when possible. They must be hard, temporally-confused, yet diverse in Allen relations questions that cover all following cases:
- Case 1: Questions with temporal expressions that mismatch with the query’s temporal expression(s): 
    - If the query specifies a span (e.g., 2005–2007), use an interval that falls completely outside it (e.g., 2008, before 2005, after 2007).
    - Adjacent years or ranges are valid negatives (e.g., query = 2010, negative = 2009).
    - Use shifted but non-overlapping intervals (e.g., query = 2010, negative = 2012–2014).
    - Include misleading implicit cues (e.g., “shortly after 2011” vs. query “in 2010”).
    - You may reuse the same passage from "positive_passages" but replace its temporal expressions with mismatched ones.
- Case 2: Questions with same temporal expression but irrelevant event/entity
    - Questions with overlapping or identical temporal expressions but targeting a different subject.
    - Example: query = “Ronaldo’s career in 2010” vs. negative = “Messi’s career in 2010”.
    - This ensures negatives are temporally aligned but semantically irrelevant.
- Case 3: "TemporalAnswer"-type questions that are either:
    - Irrelevant to the query.
    - Or ask for non-existent temporal information in the query context.
    - Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality negative passages, prioritizing quality over quantity.  
- Do not create trivial negatives (e.g., completely unrelated random text).
- Ensure no accidental overlap with valid facts in the document.

6. "temporal_query_type": Must be "Explicit", "Implicit", or "TemporalAnswer". If the temporal contains a clear date/month/year, it is "Explicit", NOT "Implicit". If the passage asks for which date/month/year of an event, it is "TemporalAnswer".
   
7. Final output: Only output valid JSON(s). Do not explain, add comments, or include extra text, since your output will be parsed automatically.

### Demonstration 1
Input:
docid: Q2465
query_id: 0
query: "On 2 July 2010 , after helping Setúbal avoid top-flight relegation , Barbosa was released by Porto , signing a three-year contract with S.C . Braga."
SUTIME's output: [{'timex-value': '2010-07-02', 'start': 3, 'end': 14, 'text': '2 July 2010', 'type': 'DATE', 'value': '2010-07-02'}, {'timex-value': 'P3Y', 'start': 111, 'end': 121, 'text': 'three-year', 'type': 'DURATION', 'value': 'P3Y'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?.

Output:
{"query_id":0,"query":"On 2 July 2010, after helping Setúbal avoid top-flight relegation, Barbosa was released by Porto, signing a three-year contract with S.C. Braga.","temporal":["2 July 2010","three-year contract"],"positive_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2010 to 2013?","temporal":["from 2010 to 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for on 2 July 2010?","temporal":["2 July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa join after leaving Porto in July 2010?","temporal":["after leaving Porto in July 2010"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa sign a three-year contract with?","temporal":["three-year contract"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"After being released by Porto, which team did Hélder Barbosa sign a contract with?","temporal":["After being released by Porto"],"allen_relation":"MetBy","temporal_query_type":"Implicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between July 2010 and July 2013?","temporal":["between July 2010 and July 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"When did Hélder Barbosa sign a contract with S.C. Braga.?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2465,"text":"How long did Hélder Barbosa's contract with S.C. Braga last?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2002 to 2009?","temporal":["from 2002 to 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between 2006 and 2009?","temporal":["between 2006 and 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for after 2014?","temporal":["after 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for during 2008?","temporal":["during 2008"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for prior to 2010?","temporal":["prior to 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Ronaldo play for in July 2010?","temporal":["in July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 1

### Demonstration 2
Input:
docid: 2466
query_id: 1
query: "Rarely used in the first months , he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish ."
SUTIME's output: [{'timex-value': 'PXM', 'start': 15, 'end': 31, 'text': 'the first months', 'type': 'DURATION', 'value': 'PXM'}, {'timex-value': '2011-01', 'start': 78, 'end': 90, 'text': 'January 2011', 'type': 'DATE', 'value': '2011-01'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?"

{"query_id":1,"query":"Rarely used in the first months, he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish .","temporal":["after the January 2011 departure of Matheus"],"positive_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for between January 2011 and December 2011?","temporal":["between January 2011 and December 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for as of January 2011?","temporal":["as of January 2011"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after Matheus departed in January 2011?","temporal":["after Matheus departed in January 2011"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa contribute goals to during the 2011 season?","temporal":["during the 2011 season"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for throughout 2011?","temporal":["throughout 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"When did Hélder Barbosa start getting more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2466,"text":"When did Hélder Barbosa begin gaining more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for before 2010?","temporal":["before 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after 2013?","temporal":["after 2013"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for in 2007?","temporal":["in 2007"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for prior to joining Braga?","temporal":["prior to joining Braga"],"allen_relation":"Before","temporal_query_type":"Implicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for during 2014?","temporal":["during 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"After the January 2011 departure of Matheus, did Ronaldo get more playing time?","temporal":["After the January 2011 departure of Matheus"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 2"""

import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")

def normalize_and_split_string_by_punctuation(text, add_title=False):    
    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)
    
    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)    

    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"
    
    # Remove : and ;
    punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    
    if add_title:
        title = text.split(": ")[0]
        return [f"{title}: " +  x.strip() for ix, x in enumerate(re.findall(punct_regex, text)) if ix != 0]
    
    return [x.strip() for x in re.findall(punct_regex, text)]



In [ ]:
a = new_train_jsonl[0] # {"query_id":14659,"query":"Clemente Palma - Life: Upon his return to Peru , he resumed his position as curator of the National Library of Peru , a post that he held until November 1911 . During this period , he founded several cultural and literary magazines such as Prisma and Variedades and the daily newspaper La Crónica . From 1911 to 1918 , he dedicated himself to the direction of these magazines . He was director of the magazines Prisma ( 1906–1908 ) and Variedades ( 1908–1931 ) and the newspaper La Crónica ( 1912–1929 ) .","positive_passages":[{"docid":"Q2281130","text":"Clemente Palma was an employee for whom between Dec 1929 and 1930?"}]}

text = a["query"]
query_id = a["query_id"]
docid = int(a["positive_passages"][0]["docid"][1:])
positive_passages = a["positive_passages"]
# negative_passages = a["negative_passages"]

positive_passages = [x['text'] for x in positive_passages]
# negative_passages = [x['text'] for x in negative_passages]

query_list = normalize_and_split_string_by_punctuation(text,add_title=True)
final_query_list = []
final_sutime_list = []

messages = []

final_query_list = []
final_sutime_list = []

buffer = []

for q in query_list:
    parsed = sutime.parse(q)

    if len(parsed) == 0:
        # No temporal expression → keep merging
        buffer.append(q)
    else:
        # Check if merging with buffer creates more temporal expressions
        if buffer:
            merged = " ".join(buffer + [q])
            if len(sutime.parse(merged)) > 1:
                temp = " ".join(buffer)
                final_query_list.append(temp)
                final_sutime_list.append(sutime.parse(temp))
                buffer = [q]
                continue
        buffer.append(q)

# Flush leftover
if buffer:
    merged = " ".join(buffer)
    temp = sutime.parse(merged)
    if len(temp) == 1:
        final_query_list.append(merged)
        final_sutime_list.append(temp)

for q, sutime_output in zip(final_query_list, final_sutime_list):    
    if not (len(sutime_output) > 0 and len(word_count_regex.findall(q)) > 5):
        continue

    for t in sutime_output:
        s, e = t["start"], t["end"]
        t["value"] = q[s:e]

    content = f"Input:\ndocid: {docid}\nquery_id: {query_id}\nquery: {q}\nSUTIME's output: {sutime_output}\nExample positive passages: {','.join(positive_passages)}\nOutput:"
    
    messages.append([
        {"role":"system", "content":prompt_template},
        {"role":"user", "content":{content},}
    ])

In [ ]:
messages

In [ ]:
output = llm.chat(
    messages=messages, 
    sampling_params=sampling_params,
    # chat_template_kwargs={"enable_thinking": True},
)

In [ ]:
temporal_answer_list = [
    # Point in time
    "what day",
    "what time",
    "what date",
    "what month",
    "what year",
    "which day",
    "which time",
    "which date",
    "which month",
    "which year",
    "when did",
    "when was",
    "when were",
    "at what time",
    "at what date",
    "at what year",
    "on what day",
    "on what date",
    "on what year",

    # Duration / span
    "how long",
    "how many days",
    "how many weeks",
    "how many months",
    "how many years",
    "for how long",
    "over what period",
    "during what years",
    "during which year",
    "for what duration",
    "in what year range",
    "between what years",

    # Frequency / recurrence
    "how often",
    "how frequently",

    # Relative temporals
    "since when",
    "until when",
    "from when",
    "from what year",
    "from what date",
    "to what year",
    "to what date",
    "up to when",
    "as of when",
    "by what year",
    "by when",
    "around when",
    "at what age",
]


def fix_implicit_temporal(passage, sutime):
    """
    Adjust temporal_query_type if 'implicit' but tagger detects explicit datetime.
    """
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Implicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    if len(sutime.parse(passage["temporal"][0].lower())) > 0:
        passage["temporal_query_type"] = TemporalQueryType.Explicit
        
    lower_passage = passage["text"].lower()
    
    # Make sure that the
    implicit_temporal = []
    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            implicit_temporal.append(passage["text"][start:start+len(t)])    
    
    if not implicit_temporal:
        return {}
    
    passage["temporal"] = implicit_temporal
    return passage


def fix_temporal_answer(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.TemporalAnswer:
        return passage

    lower_passage = passage['text'].lower()
    if any(word in lower_passage for word in temporal_answer_list):
        for word in temporal_answer_list:
            if word in lower_passage:
                lower_passage = lower_passage.replace(word, "")
        
    # Ensure that it passes the SUTIME tagger.
    if len(sutime.parse(lower_passage)) > 0:
        return {}
    else:
        passage['temporal'] = []
        passage['allen_relation'] = AllenRelation.Empty
    # print(passage)
    return passage


def fix_explicit_temporal(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Explicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    lower_passage = passage["text"].lower()
    temporal = []

    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            temporal.append(passage["text"][start:start+len(t)])

    if not temporal:
        return {}

    passage["temporal"] = temporal
    return passage


def validate_passages(passages, sutime):
    """
    Validate and filter passages.
    """
    valid = []
    
    for p in passages:
        p = fix_implicit_temporal(p, sutime)
        p = fix_explicit_temporal(p, sutime)
        p = fix_temporal_answer(p, sutime)
        
        if p:
            valid.append(p)
    return valid


def validate_temporal(js, max_temporal_expressions=3):
    lower_query = js["query"].lower()
    temporal = []
    
    # Ensure that the temporal is extracted as written from the original query
    for t in js["temporal"]:
        start = lower_query.find(t.lower())
        if start != -1:
            temporal.append(js["query"][start:start+len(t)])

    if len(temporal) == 0 or len(temporal) > max_temporal_expressions:
        return []
    return temporal



In [ ]:
temporal_jsonl = []
temp = []
for sample in output:
    sample = sample.outputs[0].text # Get the generated text
    list_of_raw_js = []
    query_set = set()
    
    for x in sample.split("\n"): # Split by newline to get multiple JSONs if any
        parts = x.split(',{"query_id"')
        if len(parts) > 0:
            json_strings = [parts[0]] + [',{"query_id"' + p for p in parts[1:]]
            list_of_raw_js.extend(json_strings)
        else:
            list_of_raw_js.append(x)
    
    for raw_js in list_of_raw_js:
        raw_js = raw_js.strip()
        
        if raw_js == "":
            continue
        try:
            js = TemporalAnnotation.model_validate_json(repair_json(raw_js, ensure_ascii=False)).model_dump()
            
            if len(js["temporal"]) == 0 or len(js["positive_passages"]) == 0 or len(js["negative_passages"]) == 0 or js["query"] in query_set:
                continue
            else:
                print(js["query"])
                js["temporal"] = validate_temporal(js)
                
                if len(js["temporal"]) == 0:
                    continue
                
                # Validate passages
                js["positive_passages"] = validate_passages(
                    js["positive_passages"], sutime
                )
                
                if len(js["positive_passages"]) == 0:
                    continue
                
                js["negative_passages"] = validate_passages(
                    js["negative_passages"], sutime
                )
                
                if len(js["negative_passages"]) == 0:
                    continue
                
                temp.append(js)
                query_set.add(js["query"])
                # positive = defaultdict(int)
                # negative = defaultdict(int)
                # allen_relation = defaultdict(int)

                # for j in js["positive_passages"]:
                #     positive[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                # for j in js["negative_passages"]:
                #     negative[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                
                # print(positive)
                # pprint(positive)
                # print(positive)
                # pprint(negative)

        except Exception as e:
            print("Error:", e)
            print("Offending JSON:", raw_js)
            continue
temporal_jsonl.extend(temp)

In [ ]:
del llm
cleanup_dist_env_and_memory()

## Post-processing

In [ ]:
temp = []
for i in range(8):
    temp += read_json(f"/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/{i}.jsonl", jsonl=True)

In [ ]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", temp, jsonl=True)

In [ ]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", jsonl=True)

In [ ]:
# temporal_jsonl
positive = defaultdict(int)
negative = defaultdict(int)
allen_relation = defaultdict(int)
for x in temp_jsonl:
    for i in x["positive_passages"]:
        positive[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
    for i in x["negative_passages"]:
        negative[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
pprint(positive)
pprint(negative)

### Check for limited positive/negative passages

In [ ]:
count = 0
new_temp_jsonl = []
for x in temp_jsonl:
    if len(x["positive_passages"]) <= 1:
        print("pos", x["query_id"])
        count += 1
        continue
    if len(x["negative_passages"]) <= 1:
        print("nega", x["query_id"])
        count += 1
        continue
    new_temp_jsonl.append(x)
print(count)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", new_temp_jsonl, jsonl=True)

### Check for positive/negative passages that only contain a reference year (2025)

In [ ]:
new_temp_jsonl2 = []
count = 0
for x in new_temp_jsonl:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if all(t):
        print(x["query_id"])
        print(x["positive_passages"])
        count += 1
    else:
        new_temp_jsonl2.append(x)
print(count)   

### Ensure that either positive and negative passages must contain at least one explicit/implicit temporal. For time-sensitve-qa, we are more flexible on each case: we only remove samples with implicit temporal expressions like: "in the remaining years", "in recent years". Cases with all temporal answers positives or negatives are preserved to ensure semantic matching is preserved.


In [ ]:
count_pos = 0
count_neg = 0
count_both = 0
count = 0
query_set = set()
new_temp_jsonl3 = []

for ix, x in enumerate(temp_jsonl):
    check_pos = False

    for i in x["positive_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_pos = True
            break 

    if check_pos == False:
        # print("pos", x["query_id"], x["query"][:10])
        count_pos += 1
        query_set.add(x["query_id"])

    check_neg = False    
    for i in x["negative_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_neg = True
            break

    if check_neg == False:
        count_neg += 1
        # print("neg", x["query_id"], x["query"][:10])
        query_set.add(x["query_id"])
        
    if check_neg == True or check_pos == True:
        query_set.add(x["query_id"])
        new_temp_jsonl3.append(x)
    
    if not check_neg and not check_pos:
        count_both += 1
        print(ix, x["query_id"], x["query"][:10], check_pos, check_neg)
    
print(count, count_pos, count_neg, count_both)

In [ ]:
new_temp_jsonl3 = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", jsonl=True)

### Check for relative temporal expressions in the queries

In [ ]:
count = 0
new_temp_jsonl4 = []

for x in new_temp_jsonl3:
    check = False

    if "now" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "now")
        count += 1
        check = True
        # break
    if "today" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "today")
        count += 1
        check = True
        # break
    if "current" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "current")
        count += 1 
        check = True
        # break
    if "yesterday" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "yesterday")
        count += 1     
        check = True
        # break    
    if not check:
        new_temp_jsonl4.append(x)
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", new_temp_jsonl4, jsonl=True)
print(count)

In [ ]:
count = 0
for x in new_temp_jsonl4:
    x["positive_passages"] = [j for j in x["positive_passages"] if "2025" not in j["text"]]

for x in new_temp_jsonl4:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if any(t):
        count += 1
print(count)   

In [ ]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", new_temp_jsonl4, jsonl=True)

## Create the corpus v2

In [ ]:
# # # # train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/train/train.jsonl", jsonl=True)
# # # # dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/train/dev.jsonl", jsonl=True)
# # # # test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/query.jsonl", jsonl=True)
# # # from wikimapper import WikiMapper

# # # # wiki_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/enwiki-20211220-pages-articles.jsonl", jsonl=True)

easy_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/train.easy.json", jsonl=True)

easy_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/dev.easy.json", jsonl=True)

easy_test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/easy/test.easy.json", jsonl=True)

hard_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/train.hard.json", jsonl=True)

hard_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/dev.hard.json", jsonl=True)

hard_test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/archive/annotated/hard/test.hard.json", jsonl=True)

train_jsonl = easy_train_jsonl + hard_train_jsonl
dev_jsonl = easy_dev_jsonl + hard_dev_jsonl
test_jsonl = easy_test_jsonl + hard_test_jsonl

# from wikimapper import WikiMapper
mapper = WikiMapper("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/index_enwiki-20220820.db")
# corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/enwiki-20211220-pages-articles.parquet")
# corpus_parquet["title"] = corpus_parquet["title"].str.lower().replace("u.s . ", "u.s. ")
# corpus_parquet["text"] = corpus_parquet["text"].str.lower().replace("u.s . ", "u.s. ")
# corpus_parquet = pl.from_pandas(corpus_parquet)

In [ ]:
wiki_url_regex = re.compile(r"([^#]+)")

def process(x):
    return mapper.url_to_id(re.sub(r"amp;", "", wiki_url_regex.match(x['idx'])[0]))

train_id = []
dev_id = []
test_id = []
    
if __name__ == '__main__':
    try:
        mp.set_start_method('fork', force=True)
        print("spawned")
    except RuntimeError:
        pass
    # from wikimapper import WikiMapper
    # mapper = WikiMapper("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/index_enwiki-20220820.db")

    with Pool(8) as pool:  # use all available cores
        train_id = list(tqdm(
            pool.imap(process, train_jsonl), 
            total=len(train_jsonl)
        ))

    with Pool(8) as pool:  # use all available cores
        dev_id = list(tqdm(
            pool.imap(process, dev_jsonl), 
            total=len(dev_jsonl)
        ))

    with Pool(8) as pool:  # use all available cores
        test_id = list(tqdm(
            pool.imap(process, test_jsonl), 
            total=len(test_jsonl)
        ))

    for i in range(3019, 3022 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q2279602"
        train_id[i] = "Q2279602"

    for i in range(3743, 3750 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q3057701"
        train_id[i] = "Q3057701"

    for i in range(9691, 9693 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q7697921"
        train_id[i] = "Q7697921"

    for i in range(17414, 17417 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q2279602"
        train_id[i] = "Q2279602"

    for i in range(18152, 18159 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q3057701"
        train_id[i] = "Q3057701"

    for i in range(24256, 24258 + 1):
        print(train_jsonl[i]["idx"])
        train_jsonl[i]["wiki_id"] = "Q7697921"
        train_id[i] = "Q7697921"

In [ ]:
all_ids = set()
all_ids.update(set(train_id))
all_ids.update(set(test_id))
all_ids.update(set(dev_id))
# len(all_ids) 4930

In [68]:
corpus_with_answers = corpus_parquet[corpus_parquet['id'].isin(all_ids) & ~corpus_parquet['id'].isnull()]

In [36]:
corpus_with_answers.index.tolist()

In [46]:
temporal_indices1 = np.load("/home/thuy0050/code/tevatron/louis/extract_temporal/temporal_indices1.npy")
temporal_indices2 = np.load("/home/thuy0050/code/tevatron/louis/extract_temporal/temporal_indices2.npy")
temporal_indices2 += 5_000_000
temporal_indices = np.concat([temporal_indices1, temporal_indices2])
answers_indices = np.load("/home/thuy0050/code/tevatron/louis/extract_temporal/answers_indices.npy")
final_indices = np.intersect1d(temporal_indices, answers_indices)

In [ ]:
non_gt_indices = np.setdiff1d(final_indices, corpus_with_answers.index.to_list())
final_indices_corpus_parquet_remove_gt_answers = corpus_parquet.iloc[non_gt_indices]
final_indices_corpus_parquet_remove_gt_answers = final_indices_corpus_parquet_remove_gt_answers[~final_indices_corpus_parquet_remove_gt_answers['id'].isnull()]

In [103]:
len(corpus_parquet), len(corpus_with_answers), len(final_indices_corpus_parquet_remove_gt_answers)

In [106]:
final_indices_corpus_parquet_remove_gt_answers.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/wiki_corpus/final_indices_corpus_remove_gt_answers.parquet")

In [107]:
final_indices_corpus_parquet_remove_gt_answers.sample(100_000 - 23009)

In [109]:
original_corpus = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/original_corpus.jsonl", jsonl=True)

In [121]:
random_samples = final_indices_corpus_parquet_remove_gt_answers.sample(300_000 - 23009)

In [124]:
import msgspec

class Doc(msgspec.Struct):
    docid: int
    text: str

docs = [
    Doc(docid=23009 + i, text=t)
    for i, t in enumerate(random_samples['text'])
]


In [126]:
corpus_v2 = original_corpus + docs

In [127]:
len(corpus_v2)

In [128]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus_v2.jsonl", corpus_v2, jsonl=True)

In [ ]:
# final_indices_corpus_parquet.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/wiki_corpus/final_indices_corpus.parquet")
# corpus_with_answers.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/wiki_corpus/corpus_with_gt_answers.parquet")

## Generate the test folder

In [ ]:
import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")


def search_paragraphs_index(
    recon_start_end_list, answer_start_end, recon_context, targets, paragraphs
):
    """Return the index of the paragraph span that contains the answer span."""
    potential_ix = []
    for ix, (s, e) in enumerate(recon_start_end_list):
        if targets in paragraphs[ix]["text"]:
            potential_ix.append(ix)
        if (
            answer_start_end[0] >= s
            and answer_start_end[1] <= e
            and targets in paragraphs[ix]["text"]
        ):
            return ix

    potential_start_end_list = [recon_start_end_list[x] for x in potential_ix]
    for ix, (s, e) in enumerate(potential_start_end_list):
        if answer_start_end[0] >= s:
            return ix
    return -1


def format_paragraph(paragraphs, art_title, idx, dedup_titles):
    par_title = paragraphs[idx]["title"]
    if dedup_titles and art_title == par_title:
        title_str = art_title
    else:
        title_str = f"{art_title} - {par_title}"
    return f"{title_str}: {paragraphs[idx]['text']}"


def extract_paragraphs_with_answers(
    paragraphs, answer_starts, answer_ends, targets, dedup_titles=True
):
    """
    Given Wikipedia-style paragraphs and answer spans (start/end indices),
    return a list of disambiguated paragraph strings that include both
    article title and section title.

    Parameters
    ----------
    paragraphs : list[dict]
        Each dict must have keys: "title", "text".
    answer_starts : list[int]
        List of answer start indices (aligned with reconstructed context).
    answer_ends : list[int]
        List of answer end indices.
    dedup_titles : bool, default=True
        If True, collapses "Title - Title" into "Title".

    Returns
    -------
    list[str] : formatted paragraphs with titles and text
    """

    title_set = set()
    start_end_list = []
    recon_context = ""
    final_paragraph_list = []

    # Build context + paragraph spans
    for par in paragraphs:
        title = par["title"]
        text = par["text"].strip()

        # Title case
        if len(title_set) == 0:
            recon_context += title.strip()
            title_set.add(title)

        # Add article/section title once
        if title not in title_set:
            recon_context += " " + title.strip() + " . "
            title_set.add(title)

        start_idx = len(recon_context)
        recon_context += " " + text.strip() + " "
        end_idx = len(recon_context)

        start_end_list.append((start_idx, end_idx))
    recon_context = recon_context.replace("  ", " ")

    # Match answers back to paragraph(s)
    for t, s, e in zip(targets, answer_starts, answer_ends):
        ix = search_paragraphs_index(
            start_end_list, (s, e), recon_context, t, paragraphs
        )
        art_title = paragraphs[0]["title"]

        if ix != -1:
            final_paragraph_list.append(
                format_paragraph(paragraphs, art_title, ix, dedup_titles)
            )
            continue

        # fallback: search in recon_context
        t_lower = t.lower()
        for ix, (_, end) in enumerate(start_end_list):
            if t_lower in recon_context[:end].lower():
                nearby_idxs = range(max(0, ix - 1), min(len(paragraphs), ix + 1))
                combined = " ".join(
                    format_paragraph(paragraphs, art_title, j, dedup_titles)
                    for j in nearby_idxs
                )
                final_paragraph_list.append(combined)
                break

    return final_paragraph_list


def normalize_and_split_string_by_punctuation(text):

    text = re.sub(r"[^\w\s]", "", text)

    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)

    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)

    return text

    # # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"

    # # Remove : and ;
    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    # return [x.strip() for x in re.findall(punct_regex, text)]

In [ ]:
sutime.parse("Which team did the player Attaphol Buspakom belong to from 1996 to 1998?")

In [ ]:
df_test = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet"
)
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
corpus_parquet["text"] = corpus_parquet["text"].apply(
    lambda x: normalize_and_split_string_by_punctuation(x.replace("  Section::::", " Section: " ).lower())
)

In [ ]:
df_test = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet"
)
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
corpus_parquet["text"] = corpus_parquet["text"].apply(
    lambda x: normalize_and_split_string_by_punctuation(x.replace("  Section::::", " Section: " ).lower())
)

output_jsonl = []
count = 0
count_unanswerable = 0
count0 = set()
test_jsonl = []

df_qrel = []

for i in range(len(df_test)):
    sample = df_test.iloc[i]

    targets = sample['targets'].tolist()
    wiki_id = sample["wiki_id"]

    corpus_rows = corpus_parquet[(corpus_parquet["id"] == wiki_id)]

    if sample["unanswerable"]:
        # df_qrel.append((i, tuple(corpus_rows.index.tolist())))
        count_unanswerable += 1
        continue

    for t in targets:
        t = normalize_and_split_string_by_punctuation(t).lower()

        index_list = corpus_rows[corpus_rows["text"].str.contains(t)].index

        if len(index_list) >= 1:
            df_qrel.append((i, tuple(index_list.tolist())))
        if len(index_list) == 0:
            if t in " ".join(corpus_rows["text"].values):
                print(
                    t,
                    wiki_id,
                    t in sample["context"],
                    t in " ".join(corpus_rows["text"].values),
                )
            count += 1
            continue
    test_jsonl.append(
        {
            "query_id": i,
            "query": sample["question"],
            "answers": sample["targets"].tolist(),
            "docid": wiki_id
        }
    )
    # count += 1
    #     count.add(wiki_id)
    # if len(index_list) == 0:
    #     count0.add(wiki_id)
    #     # print(t)
    #     # print(" ".join(corpus_rows["text"].values))
    #     # break

In [ ]:
len(test_jsonl), len(df_test), count, count_unanswerable

In [ ]:
with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/qrel.txt",
    "w",
) as outfile:
    for ix, v in enumerate(df_qrel):
        query_id = v[0]

        for docid in v[1]:
            outfile.write(f"{query_id} 0 {docid} {1}\n")

with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/qrel.txt",
    "r",
) as f:
    temp = [x for x in f.readlines()]

from natsort import natsorted
temp = natsorted(list(set(temp)), key=lambda x: x.split(" ")[0])

with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/qrel.txt",
    "w",
) as outfile:
    # for i in set(temp):
    for i in temp:
        outfile.write(i)

In [ ]:
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
corpus_jsonl = []
for i in range(len(corpus_parquet)):
    row = corpus_parquet.iloc[i]
    corpus_jsonl.append({"docid": i, "text": row["text"]})
# write_json(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.jsonl", corpus_jsonl, jsonl=True
# )

### Prepare the dev

In [ ]:
wiki_url_regex = re.compile(r"([^#]+)")


def process(x):
    return mapper.url_to_id(re.sub(r"amp;", "", wiki_url_regex.match(x["idx"])[0]))


train_id = []
dev_id = []
test_id = []

if __name__ == "__main__":
    with Pool(16) as pool:  # use all available cores
        dev_id = list(tqdm(pool.imap(process, dev_jsonl), total=len(dev_jsonl)))

In [ ]:
df_dev = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/dev.parquet"
)
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)
corpus_parquet["text"] = corpus_parquet["text"].apply(
    lambda x: normalize_and_split_string_by_punctuation(
        x.replace("  Section::::", " Section: ").lower()
    )
)
unmodified_corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/corpus.parquet"
)

In [ ]:
output_jsonl = []
count = 0
count_unanswerable = 0
count0 = set()

df_qrel = []

for i in range(len(df_dev)):
    sample = df_dev.iloc[i]

    targets = sample["targets"].tolist()
    wiki_id = sample["wiki_id"]

    corpus_rows = corpus_parquet[(corpus_parquet["id"] == wiki_id)]

    if sample["unanswerable"]:
        # df_qrel.append((i, tuple(corpus_rows.index.tolist())))
        count_unanswerable += 1
        continue

    for t in targets:
        t = normalize_and_split_string_by_punctuation(t).lower()

        index_list = corpus_rows[corpus_rows["text"].str.contains(t)].index

        # if len(index_list) >= 1:
        #     df_qrel.append((i, tuple(index_list.tolist())))
        if len(index_list) == 0:
            if t in " ".join(corpus_rows["text"].values):
                print(
                    t,
                    wiki_id,
                    t in sample["context"],
                    t in " ".join(corpus_rows["text"].values),
                )
            count += 1
            continue
        elif len(index_list) == 1:

            output_jsonl.append(
                {
                    "query_id": i,
                    "query": " ".join(
                        [unmodified_corpus_parquet.iloc[j]['text'] for j in index_list]
                    ),
                    "positive_passages": [
                        {"docid": sample["wiki_id"], "text": sample["question"]}
                    ],
                }
            )

In [ ]:
# corpus_parquet["text"].iat[13194] = 'Krasimir Balakov  Krasimir Genchev Balakov (, ; born 29 March 1966) is a Bulgarian professional football manager and former player who last served as the head coach of Bulgarian club CSKA 1948. A former attacking midfielder, he was a key member of the Bulgaria national team that finished fourth in the 1994 FIFA World Cup. He is considered as second only to Hristo Stoichkov among Bulgarian footballers of his generation.  Section::::Club career. Balakov began his club career at the local Etar Veliko Tarnovo, before transferring to Portugal\'s Sporting Clube de Portugal in 1990, playing alongside future Ballon D\'Or recipient Luís Figo, his compatriot Yordanov, and future two-time Champions League winner Paulo Sousa. Though Sporting had a quality squad, Balakov only managed to win the 1994–95 Portuguese Cup during his time at the club. In 1995, he transferred to Germany\'s VfB Stuttgart where he won two UEFA Intertoto Cups (2000 and 2002) and a DFB-Pokal (1997), before retiring in 2003 - the same year that he called time on an international career which had spanned 15 years and 92 caps. As an attacking midfielder Balakov formed a successful attacking partnership with strikers Fredi Bobic and Giovane Élber at Stuttgart. The trio were known as the "magic triangle". He was voted as Stuttgart\'s best player of all time. He stayed at Stuttgart until retiring as a player in 2003, although he did make a comeback as a player two years later when he made a single appearance as player-manager of VFC Plauen.  Section::::Coaching career. The year after he retired, Krasimir became assistant coach of the club he had just retired from, VfB Stuttgart. He stayed in this position for two years before deciding to become a player-manager at VFC Plauen, where he remained for just a short time.  He had been appointed on 16 January 2006 as a manager of Grasshopper Club Zürich to replace Hanspeter Latour who left for 1. FC Köln. Balakov managed to win the Intertoto Cup, thus qualified the club to the UEFA Cup for 2006–07 season.  He had been appointed on 29 October 2007 as a manager of FC St.  Gallen to replace Rolf Fringer.'
# corpus_parquet["text"].iat[
#     13195
# ] = "Three days before the season ended, he was fired by the club management.  In December 2008, he became manager of PFC Chernomorets Burgas in his homeland, taking over from Dimitar Dimitrov, after also having considered an offer to coach the national team of his country. On 6 December 2010, he was released from PFC Chernomorets Burgas after mutual consent, following a change in the long-term vision for the club by the owner Mitko Sabev.  On 27 May 2011, it was announced that Balakov would take over the helm of Croatian club Hajduk Split.  On 22 March 2012, Balakov was appointed the manager of 1. FC Kaiserslautern. He was sacked on 17 May 2012, after being unable to prevent Kaiserslautern's relegation to the 2. Bundesliga. He subsequently continued his career as manager in his country.  On 4 January 2018, he was announced as the new manager of Etar Veliko Tarnovo with Stanislav Genchev, Iliyan Kiryakov and Kaloyan Chakarov as first team coaches.  On 14 May 2019, he was named as the new manager of the Bulgaria national football team.  In October 2019, Balakov was replaced as manager of the national team by Georgi Dermendzhiev after resigning from his role following the backlash over his denial of alleged fan racism aimed at members of the England team in a Euro 2020 qualifying match as well as a continued string of unsatisfactory results. He took over as manager of CSKA 1948 in June 2020. In late August 2020, Balakov's duties were extended to cover the organizational management as well, with assistant Yordan Yurukov becoming more actively involved in the training process. However, the latter resigned on 22 September, leaving Balakov to be the sole one in charge of the team. In June 2021, Balakov parted ways with CSKA 1948, with the club's management thanking him for establishing the team among the stronger sides in the top division of Bulgarian football."
# corpus_parquet["text"].iat[
#     13196
# ] = "Section::::International career.  Balakov made 92 appearances for Bulgaria, between 1988 and 2003 (one of the best totals in national history) and scored 16 goals. He made his debut on 2 November 1988, in the 1–1 draw with Denmark in a qualifying match for the 1990 FIFA World Cup, coming on as a late second half substitute for Hristo Stoichkov. Other than the 1994 FIFA World Cup, he also played for his country at Euro 1996 and the 1998 FIFA World Cup. At age 37 he played in the qualifications for Euro 2004 to help his teammates qualify but retired from football before the final stage in Portugal.  Section::::Honours. Etar Veliko Tarnovo  Sporting CP  VfB Stuttgart  Bulgaria  Individual"
# corpus_parquet["text"].iat[
#     4181
# ] = "Charles B. Hoeven  Charles Bernard Hoeven (March 30, 1895 – November 9, 1980) held elective office for forty consecutive years. He was elected or re-elected eleven times to the U.S. House of Representatives to represent districts in northern Iowa. He served in Congress for 22 years (from January 3, 1943 to January 3, 1965), in the Seventy-eighth Congress and in ten succeeding Congresses.  Section::::Early life and education. Hoeven was born in Hospers, Iowa; his paternal grandparents were Dutch immigrants and his maternal grandparents were German immigrants. Hoeven attended the public schools and Alton (Iowa) High School.  During World War I, Hoeven served in England and France as a sergeant in Company D, 350th Infantry, 88th Division, and with the Intelligence Service of the First Battalion.  He received a bachelor's degree from the University of Iowa at Iowa City, in 1920 and a law degree from the University of Iowa College of Law in 1922.  Section::::Political career. Hoeven was admitted to the bar in 1922 and began to practice law in Alton, Iowa. He was elected as County Attorney of Sioux County, Iowa in 1924, and served in that position from 1925 to 1937. Then, he was elected to the Iowa Senate, where he served from 1937 to 1941, the last two years as president pro tempore.  In 1940, Hoeven ran for the Republican nomination in Iowa's 9th congressional district (which was then represented by Democrat Vincent Harrington of Sioux City). Hoeven finished a close second to Albert Swanson in the primary, who in turn lost to Harrington in the general election by fewer than 2,500 votes out of over 130,000 cast. Newspapers and others speculated that, if Hoeven had won the primary, he would have defeated Harrington. Thus, when reapportionment shifted most of the old 9th district into Iowa's 8th congressional district, Hoeven became an early front-runner for the 1942 Republican primary to run against Harrington. He won the primary, and received a significant boost when Harrington resigned his House seat and the Democratic nomination two months before the 1942 general election to serve full-time in the U.S.  Army Air Corps in England."

# corpus_parquet["text"].iat[
#     4182
# ] = "Democrats quickly nominated new candidates to serve out Harrington's 9th district term and to run against Hoeven in the 8th district, but Hoeven won the 8th district seat by over 19,000 votes.  Hoeven was then re-elected to Congress from that district an additional nine times, the last time in 1960 (when he defeated future U.S. District Court Judge Donald E. O'Brien). Following the 1960 census, Iowa lost a congressional district, and the bulk of his territory was reconfigured as the 6th district. Hoeven was elected again. He chose not to run in 1964, the year in which 48 Republican seats (including Iowa's Sixth District) were lost to Democrats. Hoeven voted in favor of the Civil Rights Acts of 1957, 1960, and 1964, as well as the 24th Amendment to the U.S. Constitution.  Hoeven also served as vice president of a savings bank.  In the Republican Party, Hoeven was a delegate to each Iowa State Republican Convention from 1925 to 1970, serving as chairman of the 1940 state convention. He was a delegate to the 1964 Republican National Convention. In 1942, he also served as temporary and permanent chairman of Iowa Republican State Judicial Convention.  Section::::Retirement and death. After retiring from Congress, Hoeven resided in Orange City, Iowa, where he died on November 9, 1980. He was interred in Nassau Township Cemetery, in Alton, Iowa."

In [ ]:
len(output_jsonl), len(df_dev), count, count_unanswerable, len(df_dev) - count_unanswerable

In [ ]:
len(dev_jsonl), len(df_dev), count, count_unanswerable, len(df_dev) - count_unanswerable

In [ ]:
temp = defaultdict(set)
new_dev_jsonl = []

for i in output_jsonl:
    temp[i["query"]].add(
        (i["positive_passages"][0]["docid"], i["positive_passages"][0]["text"])
    )

for k, v in temp.items():
    temp[k] = [{"docid":x[0], "text":x[1]} for x in v]

for ix, (k, v) in tqdm(enumerate(temp.items())):
    try:
        if list(v)[0]["docid"] is None:
            continue
        new_dev_jsonl.append(
            {
                "query_id": ix,
                "query": k,
                "positive_passages": [x for x in v],
            }
        )
    except Exception as e:
        print(v)
write_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/dev.jsonl",
    new_dev_jsonl,
    jsonl=True,
)

## Refine Qrel

In [4]:
# original_qrel = read_txt("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/qrel.txt", lines=True)
# new_qrel = read_txt("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/qrel_temp.txt", lines=True)
corpus_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/original_corpus_23009.jsonl", jsonl=True)
corpus_parquet = pd.read_parquet('/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/wiki_corpus/corpus_with_gt_answers.parquet')
test_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet")

In [58]:
mapper.title_to_id("1._FC_Köln")

In [ ]:
df_grouped['id'] = df_grouped['title'].apply(lambda x: mapper.title_to_id(x))



In [143]:
import chonkie
# from chonkie import RecursiveChunker, RecursiveRules
chunker = chonkie.SentenceChunker(tokenizer='o200k_base', chunk_size=128, min_sentences_per_chunk=1)

In [175]:
def process_row(args):
    i, title, text, row_id = args
    chunks = []

    # Split by section markers
    sections = text.split("Section::::")

    for ix, section in enumerate(sections):
        if ix == 0:
            # First part = no header
            for t in chunker.chunk(section):
                chunks.append((row_id, f"Title: {title}. {t}"))
        else:
            # Extract section header
            if ". " in section:
                header = section.split(". ")[0]
            else:
                header = ""
            section_title = f"Section: {header}."

            for jx, t in enumerate(chunker.chunk(section)):
                if t.token_count > 5:
                    prefix = section_title if jx != 0 else "Section:"
                    chunks.append((row_id, f"Title: {title}. {prefix} {t})"))

    return chunks


In [176]:
import multiprocess as mp

try:
    mp.set_start_method("spawn", force=True)
except:
    pass

def init_worker():
    global chunker
    import chonkie
    chunker = chonkie.SentenceChunker(tokenizer='o200k_base', chunk_size=128, min_sentences_per_chunk=1)
    
args_list = [
    (i, row['title'], row['text'], row['id'])
    for i, row in df_grouped.iterrows()
]

if __name__ == "__main__":
    with Pool(8, initializer=init_worker) as p:
        results = list(tqdm(p.imap(process_row, args_list), total=len(args_list)))

In [177]:
final_results = [txt for sublist in results for txt in sublist]
df = pd.DataFrame(final_results, columns=["wiki_id", "text"])

df["docid"] = list(range(len(final_results)))

In [181]:
df_test = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/test.parquet")

In [183]:
df_test = df_test[df_test['unanswerable'] == False][['question', 'targets', 'wiki_id']]

In [187]:
df

In [ ]:
joined_df = df_test.merge(df, on="wiki_id")
joined_df['targets'] = joined_df['targets'].apply(lambda x: tuple(x))
joined_df = joined_df.drop_duplicates()

In [206]:
# df.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus23009_sentencechunk128.parquet")
joined_df.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus23009_sentencechunk128_with_test_query.parquet")

In [ ]:
doc_text = "As a result of the team's 20th championship for the 2014–15 Süper Lig season, their logo hereafter contains four stars representing their 20 championships for the league; each star corresponds to five of the team's championships.  Section::::History. Galatasaray SK was founded in October 1905 (the exact day is disputed, but is traditionally accepted as \"17 Teşrinievvel 1321\" according to the Rumi calendar, which corresponds to \"30 October 1905\" according to the Gregorian calendar) by Ali Sami Yen and other students of Galatasaray High School (a high school in Istanbul which was established in 1481) as a football club. Ali Sami Yen became Galatasaray SK's first president and was given the club's membership number \"1\". The team's first match was against Cadi-Keuy FC and Galatasaray won this match with a score of 2–0. There were discussions about the club's name, in which some suggested \"Gloria\" (victory) and others \"Audace\" (courage), but it was decided that its name would be Galatasaray.  In addition to Ali Sami Yen (Club member No. 1), who was the driving force behind the club's foundation, Asim Tevfik Sonumut (2), Emin Bülent Serdaroğlu (3), Celal İbrahim (4), Boris Nikolov (5), Milo Bakić (6), Pavle Bakić (7), Bekir Sıtkı Bircan (8), Tahsin Nihat (9), Reşat Şirvanizade (10), Hüseyin Hüsnü (11), Refik Cevdet Kalpakçıoğlu (12) and Abidin Daver (13) were also involved in the decision to organize such a club.  The name Galatasaray itself comes from that of Galatasaray High School, which in turn takes its name from Galata Sarayı Enderûn-u Hümâyûn (Galata Palace Imperial School), the name of the original school founded on the site in 1481, and which in turn took its name from the nearby medieval Genoese citadel of Galata (the modern quarter of Karaköy) in the Beyoğlu (Pera) district of Istanbul. Galatasaray literally means \"Galata palace\""
pt_list = sutime.parse(doc_text)

query = "Who was the head coach of the team Galatasaray S.K. (football) from 1911 to 1914?"
qt_list = sutime.parse(query)

# Process for TempRetriever

In [ ]:
train_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl",
    jsonl=True,
)

In [ ]:
new_train_jsonl = [
    {
        "query_id": x["query_id"],
        "query": x["query"],
        "positive_passages": [i for i in x["positive_passages"] if i["temporal_query_type"] != "TemporalAnswer"],
        "negative_passages": [i for i in x["negative_passages"] if i["temporal_query_type"] != "TemporalAnswer"]
    }
    for x in train_jsonl
]

In [ ]:
new_train_jsonl2 = []
for i in new_train_jsonl:
    if len(i["positive_passages"]) == 0:
        continue 
    if len(i["negative_passages"]) == 0:
        continue 
    new_train_jsonl2.append(i)

# Dev

In [ ]:
original = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train.jsonl", jsonl=True)
original_with_temporal = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/backup/train_temporal_v3.jsonl",
    jsonl=True,
)
our = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal.jsonl",
    jsonl=True,
)

In [ ]:
our_query_id_set = set([x["query_id"] for x in our])
our_dict_to_list = defaultdict(list)

In [ ]:
for row in our:
    our_dict_to_list[row['query_id']].append(row)

In [ ]:
new_our = []
for i in range(8060):
    if i not in our_query_id_set:
        new_our.append(original[i])
    else:
        new_our.extend([x for x in our_dict_to_list[i]])

In [ ]:
len(new_our), len(our)

In [ ]:
write_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal_2.jsonl",
    new_our, jsonl=True,
)